# Session 4 · Scalable Deployment: Serving, Lite, Kubernetes and TFX

Generative AI › Track 4 · AI Architecture and Agents

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-04_savedmodel_export.py`

In [1]:
"""
Session 4 · Example 1 — Export a Keras model as a versioned SavedModel, the format TF Serving loads.

TF Serving loads folders like models/iris/1/ and answers POST /v1/models/iris:predict.
We export, reload without any Keras code (as a server would) and show the REST JSON.

Run:  python session-04_savedmodel_export.py
"""
import os, io, json, tempfile, contextlib
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import tensorflow as tf
import keras
from sklearn.datasets import load_iris

keras.utils.set_random_seed(3)
iris = load_iris()
X, y = iris.data.astype("float32"), iris.target
mean, std = X.mean(0), X.std(0)

# Put preprocessing INSIDE the model so clients can send raw measurements
model = keras.Sequential([
    keras.Input(shape=(4,), name="measurements"),
    keras.layers.Normalization(mean=mean, variance=std ** 2),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(3, activation="softmax"),
])
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X, y, epochs=200, verbose=0)
print(f"training accuracy: {model.evaluate(X, y, verbose=0)[1]:.3f}")

root = tempfile.mkdtemp()
export_dir = os.path.join(root, "models", "iris", "1")          # <model_name>/<version>
with contextlib.redirect_stdout(io.StringIO()):                  # silence the export log
    model.export(export_dir)
print("\nSavedModel folder (version 1):")
for dirpath, _, files in sorted(os.walk(export_dir)):
    rel = os.path.relpath(dirpath, root)
    for f in sorted(files):
        print(f"  {rel}/{f}")

# Reload with plain TensorFlow: no Keras class or training code needed
loaded = tf.saved_model.load(export_dir)
serve = loaded.signatures["serving_default"]
print("\nSignature inputs :", {k: v.shape for k, v in serve.structured_input_signature[1].items()})
print("Signature outputs:", {k: v.shape for k, v in serve.structured_outputs.items()})

flowers = [[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]]
out = list(serve(tf.constant(flowers, dtype=tf.float32)).values())[0].numpy()

request = {"signature_name": "serving_default", "instances": flowers}
response = {"predictions": [[round(float(v), 4) for v in row] for row in out]}
print("\nREST request body  ->", json.dumps(request))
print("REST response body <-", json.dumps(response))
for f, p in zip(flowers, out):
    print(f"  {f} -> {iris.target_names[p.argmax()]} ({p.max():.0%})")

print("\nServe it:  docker run -p 8501:8501 -v \"$PWD/models/iris:/models/iris\" -e MODEL_NAME=iris tensorflow/serving")

training accuracy: 0.973
INFO:tensorflow:Assets written to: /tmp/tmp8gi1_r27/models/iris/1/assets



SavedModel folder (version 1):
  models/iris/1/fingerprint.pb
  models/iris/1/saved_model.pb
  models/iris/1/variables/variables.data-00000-of-00001
  models/iris/1/variables/variables.index



Signature inputs : {'measurements': TensorShape([None, 4])}
Signature outputs: {'output_0': TensorShape([None, 3])}

REST request body  -> {"signature_name": "serving_default", "instances": [[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]]}
REST response body <- {"predictions": [[0.9989, 0.0011, 0.0], [0.0, 0.007, 0.9929]]}
  [5.1, 3.5, 1.4, 0.2] -> setosa (100%)
  [6.7, 3.0, 5.2, 2.3] -> virginica (99%)

Serve it:  docker run -p 8501:8501 -v "$PWD/models/iris:/models/iris" -e MODEL_NAME=iris tensorflow/serving


### Example · `session-04_tflite_convert.py`

In [2]:
"""
Session 4 · Example 2 — Convert a Keras model to TensorFlow Lite, quantise it and run it with the interpreter.

TF Lite (LiteRT) runs models on phones and microcontrollers; dynamic-range
quantisation stores weights as int8 (about 4x smaller).

Run:  python session-04_tflite_convert.py
"""
import os, io, contextlib
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import tensorflow as tf
import keras
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(5)
X, y = load_digits(return_X_y=True)
X = (X / 16.0).astype("float32")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=5, stratify=y)

model = keras.Sequential([
    keras.Input(shape=(64,)),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_tr, y_tr, epochs=25, batch_size=32, verbose=0)
keras_acc = model.evaluate(X_te, y_te, verbose=0)[1]
print(f"Keras model: {model.count_params():,} params | test accuracy {keras_acc:.4f}")


def convert(quantize):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantize:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]      # dynamic-range int8 weights
    with contextlib.redirect_stdout(io.StringIO()):
        return conv.convert()


def tflite_predict(blob, data):
    interp = tf.lite.Interpreter(model_content=blob)
    inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
    interp.resize_tensor_input(inp["index"], data.shape); interp.allocate_tensors()
    interp.set_tensor(inp["index"], data); interp.invoke()
    return interp.get_tensor(out["index"])


keras_pred = model.predict(X_te, verbose=0).argmax(1)
print(f"\n{'format':22s} {'size KB':>8s} {'accuracy':>9s} {'agrees with Keras':>18s}")
sizes = {}
for name, q in [("TFLite float32", False), ("TFLite dynamic int8", True)]:
    blob = convert(q)
    pred = tflite_predict(blob, X_te).argmax(1)
    sizes[name] = len(blob) / 1024
    print(f"{name:22s} {sizes[name]:8.1f} {np.mean(pred == y_te):9.4f} {np.mean(pred == keras_pred):17.1%}")
print(f"\nQuantised model is {sizes['TFLite float32'] / sizes['TFLite dynamic int8']:.1f}x smaller.")

# Raw weight storage, for intuition
n = model.count_params()
print(f"Weights alone: {n * 4 / 1024:.1f} KB as float32 vs {n / 1024:.1f} KB as int8")

Keras model: 50,826 params | test accuracy 0.9733

format                  size KB  accuracy  agrees with Keras


INFO:tensorflow:Assets written to: /tmp/tmpkmnjcyn3/assets


TFLite float32            200.7    0.9733            100.0%


INFO:tensorflow:Assets written to: /tmp/tmpkyb55i9e/assets


TFLite dynamic int8        57.7    0.9733            100.0%

Quantised model is 3.5x smaller.
Weights alone: 198.5 KB as float32 vs 49.6 KB as int8


### Example · `session-04_autoscaling.py`

In [3]:
"""
Session 4 · Example 3 — How a Kubernetes HorizontalPodAutoscaler sizes a model service over a day.

HPA rule: desired = ceil(pods * utilisation / target). One pod serves ~60 req/s at 100% CPU;
target 70%. We simulate a results-day spike and compare cost with a fixed fleet.

Run:  python session-04_autoscaling.py
Creates: session-04_autoscaling.png
"""
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

POD_CAPACITY_RPS = 60          # requests/second one pod can serve at 100% CPU
TARGET = 0.70                  # HPA target utilisation
MIN_PODS, MAX_PODS = 2, 20
COST_PER_POD_HOUR = 0.05       # USD, illustrative cloud price

# Requests per second each hour (results announced at 10:00 -> spike)
traffic = [20, 15, 10, 10, 12, 20, 40, 90, 160, 260, 900, 720, 480,
           350, 300, 260, 240, 260, 300, 280, 200, 120, 60, 30]

pods = MIN_PODS
history = []
print(f"{'hour':>4} {'rps':>5} {'util before':>12} {'pods':>5} {'util after':>11}")
for hour, rps in enumerate(traffic):
    util = rps / (pods * POD_CAPACITY_RPS)                       # what the HPA observes
    desired = math.ceil(pods * util / TARGET)
    pods_new = max(MIN_PODS, min(MAX_PODS, desired))
    after = rps / (pods_new * POD_CAPACITY_RPS)
    flag = "  <- capped at max, requests queue" if after > 1 else ""
    if 6 <= hour <= 14:                                          # print the busy part of the day
        print(f"{hour:4d} {rps:5d} {util:12.0%} {pods_new:5d} {after:11.0%}{flag}")
    pods = pods_new
    history.append(pods)

peak_pods = math.ceil(max(traffic) / (POD_CAPACITY_RPS * TARGET))
auto_cost = sum(history) * COST_PER_POD_HOUR
fixed_cost = min(peak_pods, MAX_PODS) * 24 * COST_PER_POD_HOUR
print(f"\nPod-hours with autoscaling: {sum(history)} -> ${auto_cost:.2f}/day")
print(f"Fixed fleet sized for peak ({min(peak_pods, MAX_PODS)} pods): {min(peak_pods, MAX_PODS) * 24} pod-hours -> ${fixed_cost:.2f}/day")
print(f"Saving: {100 * (1 - auto_cost / fixed_cost):.0f}%")

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(range(24), traffic, color="#FCD34D", label="requests/second")
ax1.set_xlabel("Hour of day"); ax1.set_ylabel("Requests per second")
ax2 = ax1.twinx()
ax2.step(range(24), history, where="mid", color="#FF6B1A", lw=2.5, label="pods")
ax2.set_ylabel("Pods running"); ax2.set_ylim(0, MAX_PODS + 2)
ax1.set_title("HPA scaling a model service through a results-day spike")
fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.88))
fig.tight_layout(); fig.savefig("session-04_autoscaling.png", dpi=110)
print("Saved chart: session-04_autoscaling.png")

hour   rps  util before  pods  util after
   6    40          33%     2         33%
   7    90          75%     3         50%
   8   160          89%     4         67%
   9   260         108%     7         62%
  10   900         214%    20         75%
  11   720          60%    18         67%
  12   480          44%    12         67%
  13   350          49%     9         65%
  14   300          56%     8         62%

Pod-hours with autoscaling: 142 -> $7.10/day
Fixed fleet sized for peak (20 pods): 480 pod-hours -> $24.00/day
Saving: 70%


Saved chart: session-04_autoscaling.png


### Example · `session-04_project_mini_tfx_pipeline.py`

In [4]:
"""
Session 4 · Mini project (full solution) — A mini TFX-style pipeline with validation gates.

Real-life problem: a lender retrains its loan-default model weekly. Broken data must never
reach production, and a new model is pushed only if it beats the serving one. Each TFX
component (ExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Trainer,
Evaluator, Pusher) is a small function with the same job. Three weeks are processed.

Run:  python session-04_project_mini_tfx_pipeline.py
"""
import os, io, json, tempfile, contextlib
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import keras

keras.utils.set_random_seed(11)
SERVING_DIR = os.path.join(tempfile.mkdtemp(), "serving", "loan_default")
FEATURES = ["age", "monthly_income", "loan_amount", "years_employed"]


def make_week(seed, n=1500, broken=False):
    r = np.random.default_rng(seed)
    age = r.integers(21, 65, n).astype(float)
    income = r.lognormal(np.log(90_000), 0.5, n)
    loan = r.uniform(50_000, 1_500_000, n)
    years = np.clip(age - 21 - r.integers(0, 5, n), 0, None)
    logit = -2.2 + 2.2 * (loan / (income * 12)) - 0.04 * years - 0.01 * (age - 40)
    default = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    X = np.column_stack([age, income, loan, years])
    if broken:
        X[:, 1] /= 1000                       # income sent in thousands
        X[r.random(n) < 0.08, 0] = np.nan     # 8% missing ages
    return X, default

# ---------------- components ----------------
def example_gen(X, y, seed):
    idx = np.random.default_rng(seed).permutation(len(y)); cut = int(0.8 * len(y))
    return {"train": (X[idx[:cut]], y[idx[:cut]]), "eval": (X[idx[cut:]], y[idx[cut:]])}

def statistics_gen(X):
    return {f: {"mean": float(np.nanmean(X[:, i])), "min": float(np.nanmin(X[:, i])),
                "max": float(np.nanmax(X[:, i])), "missing": float(np.isnan(X[:, i]).mean())}
            for i, f in enumerate(FEATURES)}

def schema_gen(stats):
    """Infer allowed ranges from the first (trusted) week, with 50% head-room."""
    return {f: {"min": s["min"] * 0.5, "max": s["max"] * 1.5, "max_missing": 0.0} for f, s in stats.items()}

def example_validator(stats, schema, ref_stats):
    anomalies = []
    for f in FEATURES:
        s, rule = stats[f], schema[f]
        if s["missing"] > rule["max_missing"]:
            anomalies.append(f"{f}: {s['missing']:.0%} missing")
        if s["min"] < rule["min"] or s["max"] > rule["max"]:
            anomalies.append(f"{f}: values {s['min']:.0f}..{s['max']:.0f} outside schema")
        drift = abs(s["mean"] - ref_stats[f]["mean"]) / (abs(ref_stats[f]["mean"]) + 1e-9)
        if drift > 0.25:
            anomalies.append(f"{f}: mean drifted {drift:.0%}")
    return anomalies

def transform(train_X):
    mu, sd = train_X.mean(0), train_X.std(0)
    return keras.layers.Normalization(mean=mu, variance=sd ** 2)   # becomes part of the model

def trainer(splits):
    X, y = splits["train"]
    model = keras.Sequential([keras.Input((4,)), transform(X),
                              keras.layers.Dense(16, activation="relu"),
                              keras.layers.Dense(1, activation="sigmoid")])
    model.compile("adam", "binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    model.fit(X, y, epochs=30, batch_size=64, verbose=0)
    return model

def evaluator(model, splits, baseline_auc, min_auc=0.70):
    X, y = splits["eval"]
    auc = float(model.evaluate(X, y, verbose=0)[1])
    blessed = auc >= min_auc and auc >= baseline_auc - 0.01      # small tolerance
    return auc, blessed

def pusher(model, version):
    path = os.path.join(SERVING_DIR, str(version))
    with contextlib.redirect_stdout(io.StringIO()):
        model.export(path)
    return path

# ---------------- pipeline run ----------------
ref_stats, schema, serving = None, None, {"version": 0, "auc": 0.0}
weeks = [("Week 1", make_week(1)), ("Week 2", make_week(2)), ("Week 3", make_week(3, broken=True))]
for name, (X, y) in weeks:
    print(f"=== {name}: {len(y)} applications, default rate {y.mean():.1%}")
    stats = statistics_gen(X)
    if schema is None:
        ref_stats, schema = stats, schema_gen(stats)
        print("  SchemaGen        : schema inferred from this trusted batch")
    anomalies = example_validator(stats, schema, ref_stats)
    if anomalies:
        print("  ExampleValidator : BLOCKED ->", "; ".join(anomalies))
        print(f"  Pusher           : skipped, production stays on v{serving['version']}\n")
        continue
    print("  ExampleValidator : no anomalies")
    splits = example_gen(X, y, seed=len(name))
    model = trainer(splits)
    auc, blessed = evaluator(model, splits, serving["auc"])
    print(f"  Trainer/Evaluator: eval AUC {auc:.3f} vs serving {serving['auc']:.3f} -> {'BLESSED' if blessed else 'not blessed'}")
    if blessed:
        version = serving["version"] + 1
        pusher(model, version)
        serving = {"version": version, "auc": auc}
        print(f"  Pusher           : exported SavedModel version {version}")
    print()

print("Versions in the serving folder:", sorted(os.listdir(SERVING_DIR)))
print("Production model:", json.dumps({"version": serving["version"], "eval_auc": round(serving["auc"], 3)}))

=== Week 1: 1500 applications, default rate 27.2%
  SchemaGen        : schema inferred from this trusted batch
  ExampleValidator : no anomalies


  Trainer/Evaluator: eval AUC 0.780 vs serving 0.000 -> BLESSED
INFO:tensorflow:Assets written to: /tmp/tmp2x5tqze1/serving/loan_default/1/assets


  Pusher           : exported SavedModel version 1

=== Week 2: 1500 applications, default rate 27.7%
  ExampleValidator : no anomalies


  Trainer/Evaluator: eval AUC 0.846 vs serving 0.780 -> BLESSED
INFO:tensorflow:Assets written to: /tmp/tmp2x5tqze1/serving/loan_default/2/assets


  Pusher           : exported SavedModel version 2

=== Week 3: 1500 applications, default rate 28.1%
  ExampleValidator : BLOCKED -> age: 8% missing; monthly_income: values 20..628 outside schema; monthly_income: mean drifted 100%
  Pusher           : skipped, production stays on v2

Versions in the serving folder: ['1', '2']
Production model: {"version": 2, "eval_auc": 0.846}
